# ILPCSR — Two-Stage Novita AI Summarization

Generates structured summaries for ILPCSR cases that are missing `Summary` fields.  
**Stage 1**: Compress any section exceeding `SECTION_COMPRESS_CHARS` characters (lossless — all facts/dates/amounts/citations preserved, only verbosity stripped). Compressed text is held in memory only, never written to disk.  
**Stage 2**: Feed compressed sections through the standard structured-summary prompt to produce the 5 DB fields.  
Only `case_json['Summary']` is written back to the corpus JSON file.

In [ ]:
# Cell 1 — Setup
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
from typing import List

# NB_DIR: Jupyter sets cwd to notebook directory; use that as anchor
NB_DIR = Path.cwd()
load_dotenv()

# --- Constants ---
NOVITA_BASE_URL = "https://api.novita.ai/v3/openai"
NOVITA_API_KEY_ENV = "NOVITA_API_KEY"
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
SECTION_COMPRESS_CHARS = 100_000  # compress any section exceeding this; target output is also ~100K chars

TEXT_KEYS = [
    "Fact",
    "Issue",
    "Petitioner's Argument",
    "Respondent's Argument",
    "Analysis of the law",
    "Precedent Analysis",
    "Court's Reasoning",
    "Conclusion",
    "Court Disclosure",
    "Uncategorized",
]

SUMMARY_KEYS = ["key_facts", "legal_issues", "holding", "key_statutes_cited", "key_precedents_cited"]

class LLMCaseSummary(BaseModel):
    """Structured summary of core legal arguments and decisions."""
    key_facts: str = Field(..., description="A concise, one-to-two-sentence summary of the essential facts of the case.")
    legal_issues: List[str] = Field(..., description="List as many core and subsidiary legal questions as possible that the court had to address.")
    holding: str = Field(..., description="The court's final decision or ruling, stated clearly in one or two sentences.")

client = OpenAI(
    base_url=NOVITA_BASE_URL,
    api_key=os.environ[NOVITA_API_KEY_ENV],
)

print(f"Novita client ready. Model: {MODEL}")
SECTION_COMPRESS_CHARS = 100_000  # compress any section exceeding this; target output is also ~100K chars


In [ ]:
# Cell 2 — Identify unsummarized cases (no Summary dict at all)
CORPUS_DIR = NB_DIR.parent / "il-pcsr_dataset" / "corpus"

unsummarized = []
total = 0
for f in sorted(CORPUS_DIR.glob("*.json")):
    total += 1
    case = json.loads(f.read_text(encoding="utf-8"))
    if not isinstance(case.get("Summary"), dict):
        unsummarized.append(f)

print(f"Total corpus cases : {total}")
print(f"Need summaries     : {len(unsummarized)}")
print()
for p in unsummarized:
    print(" ", p.name)

In [ ]:
# Cell 3 — Stage 1: per-section compression
# Scout context = 131K tokens. SECTION_MAX_CHARS=100K chars (~25K tokens/section).
# Chunks compressed in parallel. No retries.

import time
from concurrent.futures import ThreadPoolExecutor

CHUNK_SIZE = 120_000
SECTION_COMPRESS_CHARS = 100_000
SECTION_MAX_CHARS = 100_000
MAX_WORKERS = 4

COMPRESS_SYSTEM_PROMPT = (
    "You are a lossless legal text compressor. Compress the provided text while retaining "
    "100% of its informational content. "
    "Information preservation is the only priority — readable prose and paragraph structure "
    "are not required. You may use dense, note-like output (fragments, lists) as long as "
    "every fact is retained in full, readable words — do not abbreviate "
    "terms or names in ways that obscure their meaning.\n"
    "MUST KEEP: every party name, date, monetary figure, statute number, section reference, "
    "case citation, legal argument, procedural step, court finding, and ratio decidendi.\n"
    "REMOVE: filler phrases ('it is submitted that', 'it is pertinent to note', 'as stated "
    "above'), redundant restatements of the same point, verbose transitions, and repetitive "
    "paragraph-level summaries of content already stated.\n"
    "Output only the compressed text. No preamble, no commentary."
)

FINAL_COMPRESS_SYSTEM_PROMPT = (
    "You are a lossless legal text compressor. Compress the provided text to fit within "
    "approximately 100,000 characters while retaining 100% of its informational content. "
    "Information preservation is the only priority — readable prose and paragraph structure "
    "are not required. You may use dense, note-like output (fragments, lists) as long as "
    "every fact is retained in full, readable words — do not abbreviate "
    "terms or names in ways that obscure their meaning.\n"
    "MUST KEEP: every party name, date, monetary figure, statute number, section reference, "
    "case citation, legal argument, procedural step, court finding, and ratio decidendi.\n"
    "REMOVE: filler phrases ('it is submitted that', 'it is pertinent to note', 'as stated "
    "above'), redundant restatements of the same point, verbose transitions, and repetitive "
    "paragraph-level summaries of content already stated.\n"
    "Output only the compressed text. No preamble, no commentary."
)


def _call_novita(system_prompt: str, user_text: str, label: str) -> str:
    """Single Novita call. Raises immediately on any failure."""
    est_tokens = (len(system_prompt) + len(user_text)) // 4
    print(f"      [{label}] ~{est_tokens:,} tok ...", end=" ", flush=True)
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text},
        ],
        temperature=0.1,
    )
    result = resp.choices[0].message.content.strip()
    print(f"{time.time()-t0:.1f}s | {len(result):,} chars out")
    return result


def compress_section(section_label: str, section_text: str) -> str:
    """
    Three-step compression:
    1. Split into CHUNK_SIZE chunks at paragraph boundaries
    2. Compress all chunks in parallel
    3. Concatenate; if result > SECTION_MAX_CHARS, do a final compression pass
    """
    if len(section_text) <= SECTION_COMPRESS_CHARS:
        return section_text

    chunks = []
    remaining = section_text
    while len(remaining) > CHUNK_SIZE:
        split_at = remaining.rfind("\n", 0, CHUNK_SIZE)
        if split_at == -1:
            split_at = CHUNK_SIZE
        chunks.append(remaining[:split_at])
        remaining = remaining[split_at:].lstrip("\n")
    if remaining:
        chunks.append(remaining)

    print(f"    '{section_label}': {len(section_text):,} chars (~{len(section_text)//4:,} tok) -> {len(chunks)} chunks [parallel]")
    t_section = time.time()

    def _compress_chunk(args):
        idx, chunk = args
        return idx, _call_novita(
            COMPRESS_SYSTEM_PROMPT,
            f"### {section_label} (part {idx+1}/{len(chunks)}) ###\n{chunk}",
            f"chunk {idx+1}/{len(chunks)}"
        )

    results = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(chunks))) as pool:
        for idx, result in pool.map(_compress_chunk, enumerate(chunks)):
            results[idx] = result

    joined = "\n".join(results)
    print(f"    chunks done in {time.time()-t_section:.1f}s | joined: {len(joined):,} chars (~{len(joined)//4:,} tok)")

    if len(joined) > SECTION_MAX_CHARS:
        print(f"    final pass (>{SECTION_MAX_CHARS:,} chars) ...")
        joined = _call_novita(
            FINAL_COMPRESS_SYSTEM_PROMPT,
            f"### {section_label} ###\n{joined}",
            "final pass"
        )

    return joined

print(f"compress_section() ready | parallel={MAX_WORKERS} workers | no timeout | no retries")

In [ ]:
# Cell 4 — Stage 2: structured summary + main loop

import re

SUMMARY_SYSTEM_PROMPT = (
    "You are a specialized legal AI assistant. Your sole purpose is to read a court "
    "judgment and generate a structured summary that perfectly matches the provided JSON schema. "
    "You MUST adhere to the schema. Do not extract other facts or create your own keys."
)

SUMMARY_USER_TEMPLATE = """\
You MUST generate a JSON object that conforms exactly to the following structure.
Do not add, omit, or change any keys. Follow the example provided.

### EXAMPLE ###
Input Text: "The appellant, John Doe, was convicted under Section 307 for attempted murder after an altercation. The key issue was whether the act constituted a 'heinous crime' for bail purposes. The court referenced the landmark case of 'State v. Smith' which defined heinous crimes. The appeal was ultimately dismissed."

Output JSON:
{{
    "key_facts": "The appellant was convicted for attempted murder following an altercation.",
    "legal_issues": ["Whether the act constituted a 'heinous crime' for the purpose of denying bail."],
    "holding": "The court dismissed the appeal, implying the act was considered heinous."
}}
### END OF EXAMPLE ###

Now, analyze the following court case text and generate the corresponding JSON object.

### CASE TEXT ###
{case_text}
### END OF CASE TEXT ###
"""


def extract_json(text: str) -> str:
    """Strip markdown code fences and extract the JSON object from model output."""
    text = text.strip()
    # Strip ```json ... ``` or ``` ... ``` fences
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    # Find first { ... } block
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1:
        return text[start:end+1]
    return text


def generate_summary(case_json: dict, compressed_sections: dict) -> dict | None:
    """
    Assembles compressed sections, calls Novita for 3 LLM fields,
    appends statutes/precedents from case metadata. Returns 5-key Summary dict or None.
    """
    parts = []
    for key in TEXT_KEYS:
        text = compressed_sections.get(key, "").strip()
        if text:
            parts.append(f"[{key}]\n{text}")
    case_text = "\n\n".join(parts)

    user_prompt = SUMMARY_USER_TEMPLATE.format(case_text=case_text)
    raw_response = ""
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.1,
            # Note: Scout on Novita does not support response_format=json_object
        )
        raw_response = resp.choices[0].message.content.strip()
        json_str = extract_json(raw_response)
        validated = LLMCaseSummary.model_validate_json(json_str)
    except ValidationError as e:
        print(f"  [ERROR] Pydantic validation failed: {e}")
        print(f"  Raw output: {raw_response[:500]}")
        return None
    except Exception as e:
        print(f"  [ERROR] Novita API call failed: {e}")
        return None

    return {
        "key_facts": validated.key_facts,
        "legal_issues": validated.legal_issues,
        "holding": validated.holding,
        "key_statutes_cited": case_json.get("relevant_statutes") or [],
        "key_precedents_cited": case_json.get("relevant_precedents") or [],
    }


# --- Main loop ---
succeeded = []
failed = []

for path in unsummarized:
    print(f"\n{'='*60}")
    print(f"Processing: {path.name}")

    case_json = json.loads(path.read_text(encoding="utf-8"))

    # Stage 1 — compress each section in memory
    compressed = {}
    total_before = total_after = 0
    for key in TEXT_KEYS:
        raw_paragraphs = case_json.get(key, [])
        if not raw_paragraphs:
            continue
        raw_text = "\n".join(
            p["text"] if isinstance(p, dict) else str(p)
            for p in raw_paragraphs
        ).strip()
        if not raw_text:
            continue
        total_before += len(raw_text)
        comp_text = compress_section(key, raw_text)
        total_after += len(comp_text)
        compressed[key] = comp_text
        action = "compressed" if len(comp_text) < len(raw_text) else "passed through"
        print(f"  {key:<30} {len(raw_text):>8,} -> {len(comp_text):>8,} chars  [{action}]")

    print(f"  Total: {total_before:,} -> {total_after:,} chars ({100*(1-total_after/max(total_before,1)):.1f}% reduction)")

    # Stage 2 — generate structured summary
    print("  Generating structured summary...")
    summary_dict = generate_summary(case_json, compressed)

    if summary_dict is None:
        print(f"  [FAIL] {path.name}")
        failed.append(path.name)
        continue

    # Write only Summary back to corpus JSON
    case_json["Summary"] = summary_dict
    path.write_text(json.dumps(case_json, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  [OK] Summary written to {path.name}")
    succeeded.append(path.name)

print(f"\n{'='*60}")
print(f"Done. {len(succeeded)} succeeded, {len(failed)} failed.")
if failed:
    print("Failed cases:")
    for name in failed:
        print(f"  {name}")

In [ ]:
# Cell 5 — Verification: re-audit corpus summary completeness
from pathlib import Path
import json

CORPUS_DIR_V = Path.cwd().parent / "il-pcsr_dataset" / "corpus"
keys = ["key_facts", "legal_issues", "holding", "key_statutes_cited", "key_precedents_cited"]

total_v = has_any = has_all5 = 0
sub = {k: 0 for k in keys}

for f in CORPUS_DIR_V.glob("*.json"):
    total_v += 1
    d = json.loads(f.read_text(encoding="utf-8"))
    s = d.get("Summary")
    if not isinstance(s, dict) or not s:
        continue
    has_any += 1
    all5 = True
    for k in keys:
        if s.get(k):
            sub[k] += 1
        else:
            all5 = False
    if all5:
        has_all5 += 1

print(f"Total: {total_v} | Any Summary: {has_any} | All 5 fields: {has_all5}")
for k, v in sub.items():
    print(f"  {k}: {v}")